In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

#Load the datasets
loans = pd.read_csv('/content/drive/MyDrive/Colab_Notebooks/risk_analysis_project/loan_applications.csv')
borrowers = pd.read_csv('/content/drive/MyDrive/Colab_Notebooks/risk_analysis_project/borrower_profiles.csv')

print(loans.head())




In [ ]:
#Merging the 2 datasets
combined_df = pd.merge(loans,borrowers, on = "borrower_id", how = "inner")
print(len(combined_df))

In [ ]:
#checking for null and mismatched data types
combined_df.info()


In [ ]:
#changing data types
combined_df['application_date'] = pd.to_datetime(combined_df['application_date'])
combined_df.info()

In [ ]:
#statistical summary
combined_df.describe()

In [ ]:
#Outlier handling
high_dti = combined_df[combined_df['dti_ratio'] > 100]
print(high_dti)
print(sum(high_dti['loan_status'] == 'Default' ))

In [ ]:
high_dti.groupby('loan_status')['credit_score'].mean()

In [ ]:
plt.figure(figsize=(12,6))

sns.scatterplot(
    data = combined_df,
    x = 'dti_ratio',
    y = 'credit_score',
    hue = 'loan_status',
    alpha = 0.6
)

plt.title('Relationship between DTI Ratio, Credit Score, and Loan Status')
plt.xlabel('DTI Ratio')
plt.ylabel('Credit_score')
plt.legend(title='Loan Status', bbox_to_anchor = (1.05, 1), loc='upper left')
plt.savefig('dti_credit_loan.png',dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
bins = [520, 599, 649, 699, 749, 850]

labels = ['Fair', 'Good' , 'Great', 'Well done', 'Exceptional']

combined_df['Credit_category'] = pd.cut(combined_df['credit_score'], bins=bins, labels=labels)


In [ ]:
plt.figure(figsize=(12,6))

sns.countplot(
    data = combined_df,
    x = 'Credit_category',
    hue = 'loan_status',
    palette = 'viridis'
)

plt.title('Distribution of borrowers by credit category')
plt.xlabel('Credit score category')
plt.ylabel('Number of Borrowers')

plt.savefig('borrow_credit.png',dpi=300, bbox_inches='tight')
plt.show()





In [ ]:
bins = 0, 20 , 40 ,60 ,80 , 100, 180

labels = 'very low risk', 'low risk', 'moderate risk', 'risk', 'high risk', 'very high risk'

combined_df['dti_category'] = pd.cut(combined_df['dti_ratio'],bins=bins, labels=labels)

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(12,6))

sns.barplot(
    data=combined_df,
    x = 'dti_category',
    y = 'defaulted',
    hue = 'dti_category',
    legend=False,
    ci = None
)

plt.axhline(0.20, color = 'black', linestyle='--')

plt.title('dti - default chart')
plt.xlabel('dti category')
plt.ylabel("default rate (%)")
plt.savefig('dti-default.png',dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(12,6))

sns.barplot(
    data = combined_df,
    x = 'employment_status',
    y = 'defaulted',
    hue = 'employment_status',
    errorbar = None
)


plt.title('employment - defaulted chart')
plt.xlabel('employment_status')
plt.ylabel('defaulted (%)')
plt.savefig('employment_default.png',dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(18,12))

sns.barplot(
    data= combined_df,
    x = 'loan_purpose',
    y = 'defaulted',
    hue = 'loan_purpose',
    errorbar = None
)

plt.title('Loan purpose - Defaulted chart')
plt.xlabel('Loan purpose')
plt.ylabel('Defaulted (%)')
plt.savefig('loan_default.png',dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
numeric_df = combined_df.select_dtypes(include=['number'])

corr_matrix = numeric_df.corr()

plt.figure(figsize=(12,8))
sns.heatmap(corr_matrix, annot=True, cmap = 'coolwarm', fmt=".2f")

plt.title('Correlation heatmap of loan feature')
plt.savefig('correlation_heatmap',dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
correlation = numeric_df.corr()['defaulted'].sort_values(ascending=False)

correlation = correlation.drop(['defaulted', 'days_delinquent'])

correlation.plot(kind = 'barh', figsize=(12,8), color='skyblue')

plt.title('features most correlated with loan default')
plt.savefig('targeted_corr.png',dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Checking to see if loan amount differ significantly between defaulted and
# non-defaulted loans.
combined_df.groupby('defaulted')['loan_amount'].mean()

In [ ]:
plt.figure(figsize=(12,6))

sns.pointplot(
    data = combined_df,
    x = 'employment_status',
    y = 'defaulted',
    errorbar=None
)

plt.title('Relationship between employment status and defaulted')
plt.xlabel('employment status')
plt.ylabel('defaulted')
plt.savefig('employ_default_loan.png',dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# creating ERD of the datasets.

import graphviz

graph = graphviz.Digraph('ERD', comment = 'Loan Data Scheme', node_attr = {'shape':'record', 'color': '#2c3e50', 'fontname': 'Helvetica'})

graph.node('Borrowers', '''{ <pk> Borrower Profiles (Table) |
      borrowe_id (PK) |
      age |
      state |
      education |
      employment status |
      years_employed |
      annual Income |
      credit_score |
      home_ownership |
      dependents |
      existing_monthly_debt


}''')

graph.node('Loans', '''{<pk> Loan Applications (Table) |
      loan_id (PK) |
      borrower_id (FK) |
      application_data |
      loan_purpose |
      loan_amount |
      term months |
      interest_rate |
      monthly_payment |
      dti_ratio |
      loan_status |
      days_delinquent |
      defaulted
}''')

graph.edge('Borrowers:pk', 'Loans: fk', Label= '1:N', arrowhead="crow", arrowtail = 'None', dir='both')

graph


In [ ]:
graph.render('loan_erd', format='png', cleanup = True)



In [ ]:
#Averaging interest rate by grouping defaulted
combined_df.groupby('defaulted')['interest_rate'].mean()